In [2]:
%pwd

'd:\\first_legal_ai\\AI-Based-Legal-Chatbot\\research'

In [3]:
import os
os.chdir("../")

In [4]:
%pwd

'd:\\first_legal_ai\\AI-Based-Legal-Chatbot'

In [6]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

ImportError: cannot import name 'PyPDFLoader' from 'langchain.document_loaders' (c:\Users\USER\AppData\Local\Programs\Python\Python313\Lib\site-packages\langchain\document_loaders\__init__.py)

In [45]:
# Extract text from PDF files
def load_pdf_files(data):
    loader = DirectoryLoader(
        data,
        glob="*.pdf",
        loader_cls=PyPDFLoader
    )

    documents = loader.load()
    return documents


In [46]:
extracted_data = load_pdf_files("data")

In [47]:
extracted_data

[Document(metadata={'producer': 'Adobe PDF Library 9.9', 'creator': 'Adobe InDesign CS5.5 (7.5)', 'creationdate': '2017-05-04T12:01:01-04:00', 'moddate': '2017-05-05T14:58:51-04:00', 'trapped': '/False', 'source': 'data\\01. International Law Hadbook Collection of Instruments Book 1 Author Welcome to the United Nations.pdf', 'total_pages': 681, 'page': 0, 'page_label': 'i'}, page_content='INTERNATIONAL LAW HANDBOOK\nCOLLECTION OF INSTRUMENTS\nBOOK ONE'),
 Document(metadata={'producer': 'Adobe PDF Library 9.9', 'creator': 'Adobe InDesign CS5.5 (7.5)', 'creationdate': '2017-05-04T12:01:01-04:00', 'moddate': '2017-05-05T14:58:51-04:00', 'trapped': '/False', 'source': 'data\\01. International Law Hadbook Collection of Instruments Book 1 Author Welcome to the United Nations.pdf', 'total_pages': 681, 'page': 1, 'page_label': 'ii'}, page_content='The photograph on the cover is of a stained \nglass window in the United Nations \nHeadquarters building in New Y ork. The \nstaff of the United Nat

In [48]:
len(extracted_data)

793

In [50]:
from typing import List
from langchain.schema import Document

def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    """
    Given a list of Document objects, return a new list of Document objects
    containing only 'source' in metadata and the original page_content.
    """
    minimal_docs: List[Document] = []
    for doc in docs:
        src = doc.metadata.get("source")
        minimal_docs.append(
            Document(
                page_content=doc.page_content,
                metadata={"source": src}
            )
        )

    return minimal_docs

In [51]:
minimal_docs = filter_to_minimal_docs(extracted_data)

In [52]:
minimal_docs

[Document(metadata={'source': 'data\\01. International Law Hadbook Collection of Instruments Book 1 Author Welcome to the United Nations.pdf'}, page_content='INTERNATIONAL LAW HANDBOOK\nCOLLECTION OF INSTRUMENTS\nBOOK ONE'),
 Document(metadata={'source': 'data\\01. International Law Hadbook Collection of Instruments Book 1 Author Welcome to the United Nations.pdf'}, page_content='The photograph on the cover is of a stained \nglass window in the United Nations \nHeadquarters building in New Y ork. The \nstaff of the United Nations and Marc Chagall \ndonated the stained glass panel designed \nby the French artist as a memorial to Dag \nHammarskjöld and 15 others who died in a \nplane crash while on a peace mission in the \nCongo in 1961. Dag Hammarskjöld served as \nthe second Secretary-General of the United \nNations from 10 April 1953 until his death \non 18 September 1961. He introduced the \nconcept of peacekeeping and was awarded \nthe Nobel Peace Prize. He also defined the \nrole o

In [53]:
# Split the documents into smaller chunks
def text_split(minimal_docs):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=20,
    )
    texts_chunk = text_splitter.split_documents(minimal_docs)
    return texts_chunk

In [54]:
texts_chunk = text_split(minimal_docs)
print(f"Number of chunks: {len(texts_chunk)}")

Number of chunks: 6006


In [55]:
texts_chunk

[Document(metadata={'source': 'data\\01. International Law Hadbook Collection of Instruments Book 1 Author Welcome to the United Nations.pdf'}, page_content='INTERNATIONAL LAW HANDBOOK\nCOLLECTION OF INSTRUMENTS\nBOOK ONE'),
 Document(metadata={'source': 'data\\01. International Law Hadbook Collection of Instruments Book 1 Author Welcome to the United Nations.pdf'}, page_content='The photograph on the cover is of a stained \nglass window in the United Nations \nHeadquarters building in New Y ork. The \nstaff of the United Nations and Marc Chagall \ndonated the stained glass panel designed \nby the French artist as a memorial to Dag \nHammarskjöld and 15 others who died in a \nplane crash while on a peace mission in the \nCongo in 1961. Dag Hammarskjöld served as \nthe second Secretary-General of the United \nNations from 10 April 1953 until his death'),
 Document(metadata={'source': 'data\\01. International Law Hadbook Collection of Instruments Book 1 Author Welcome to the United Natio

In [56]:
from langchain.embeddings import HuggingFaceEmbeddings

def download_embeddings():
    """
    Download and return the HuggingFace embeddings model.
    """
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    embeddings = HuggingFaceEmbeddings(
        model_name=model_name
    )
    return embeddings

embedding = download_embeddings()

In [57]:
embedding

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [58]:
vector = embedding.embed_query("hello world")
vector

[-0.03447727486491203,
 0.03102317824959755,
 0.006734970025718212,
 0.026108985766768456,
 -0.03936202451586723,
 -0.16030244529247284,
 0.06692401319742203,
 -0.006441489793360233,
 -0.0474504791200161,
 0.014758856035768986,
 0.07087527960538864,
 0.05552763119339943,
 0.019193334504961967,
 -0.026251312345266342,
 -0.01010954286903143,
 -0.02694045566022396,
 0.022307461127638817,
 -0.022226648405194283,
 -0.14969263970851898,
 -0.017493007704615593,
 0.00767625542357564,
 0.05435224249958992,
 0.0032543970737606287,
 0.031725890934467316,
 -0.0846213847398758,
 -0.02940601296722889,
 0.05159561336040497,
 0.04812406003475189,
 -0.0033148222137242556,
 -0.058279167860746384,
 0.04196927323937416,
 0.022210685536265373,
 0.1281888335943222,
 -0.022338971495628357,
 -0.011656315997242928,
 0.06292839348316193,
 -0.032876335084438324,
 -0.09122604131698608,
 -0.031175347045063972,
 0.0526994913816452,
 0.04703482985496521,
 -0.08420311659574509,
 -0.030056199058890343,
 -0.02074483036

In [18]:
print("vec: ", len(vector))

vec:  384


In [85]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [88]:
PINECONE_API_KEY= os.getenv("PINECONE_API_KEY")
OPENAI_API_KEY= os.getenv("OPENAI_API_KEY")

os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

In [87]:
from pinecone import Pinecone
pinecone_api_key = PINECONE_API_KEY

pc = Pinecone(api_key=pinecone_api_key)

In [83]:
pc

In [64]:
from pinecone import ServerlessSpec

index_name = "legal-chatbot"

if not pc.has_index(index_name):
    pc.create_index(
        name = index_name,
        dimension=384,  # Dimension of the embeddings
        metric="cosine",  # Cosine similarity
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

index = pc.Index(index_name)

In [65]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=texts_chunk,
    embedding=embedding,
    index_name=index_name
)


In [66]:
# Load Existing index
from langchain_pinecone import PineconeVectorStore
# Embed each chunk and upsert the embeddings into your Pinecone index.
docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embedding
)

In [67]:
dswith = Document(
    page_content="University of Virginia School of Law is a youtube channel that provides tutorials on various topics.",
    metadata={"source":"Youtube"}
)

In [68]:
docsearch.add_documents(documents=[dswith])

['b065e6fb-3a89-422b-bd70-68933fc27185']

In [94]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k":3})

In [95]:
retrieved_docs = retriever.invoke("what is artical?")
retrieved_docs

[Document(id='646d39c6-21c7-4645-98d1-4a5232615624', metadata={'source': 'data\\01. International Law Hadbook Collection of Instruments Book 1 Author Welcome to the United Nations.pdf'}, page_content='tion relating to the rights and obligations of natural or juridical persons.'),
 Document(id='dd9e8908-4fcf-41fc-b39c-7f5414241b8b', metadata={'source': 'data\\01. International Law Hadbook Collection of Instruments Book 1 Author Welcome to the United Nations.pdf'}, page_content='tion relating to the rights and obligations of natural or juridical persons.'),
 Document(id='d57bfbff-2c17-4ddb-92e0-8c76e1ecc5a0', metadata={'source': 'data\\01. International Law Hadbook Collection of Instruments Book 1 Author Welcome to the United Nations.pdf'}, page_content='and artistic life and shall encourage the provision of appropriate and equal opportunities for cul -\ntural, artistic, recreational and leisure activity.')]

In [89]:
from langchain_openai import ChatOpenAI

chatModel = ChatOpenAI(model="gpt-4o")

In [90]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [91]:
system_prompt = (
    "You are an assistant for question-answering tasks."
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [92]:
question_answer_chain = create_stuff_documents_chain(chatModel, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)